In [ ]:
#import packages
from sklearn.metrics import classification_report, accuracy_score
from keras.models import Sequential
from keras.layers import Activation, Dense, Dropout, LSTM, Bidirectional, GRU, SimpleRNN, Flatten, Embedding
from keras.layers.convolutional import Conv1D, MaxPooling1D
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from sklearn import preprocessing
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import precision_score
from sklearn.metrics import accuracy_score
import seaborn as sns
import pandas as pd
import numpy as np
from ta import add_all_ta_features # Library that does financial technical analysis 

#to plot within notebook
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

#for normalizing data
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler(feature_range=(0, 1))

In [ ]:
# Importing the training set
df = pd.read_csv('Data/ATTIJARIWAFA-BANK.csv', index_col="Date", parse_dates=True)

# Add all technical analysis to the dataframe we've already loaded
df = add_all_ta_features(df, "Open", "High", "Low", "Close", "Volume", fillna=True)

print(df)

In [ ]:
# Visualize Close stock prices
df.plot.line(y="Close", use_index=True)

In [ ]:
#how many days data will be used to create series to train RNN
SERIES_LENGTH=20

In [ ]:
def scale_data(df):
    for column in df.columns:
        df[column] = preprocessing.scale(df[column].values)
    return df

In [ ]:
import numpy as np
def process_data(df):
    df["Label"] = df.rolling(5).apply(lambda x: x.iloc[1] > x.iloc[0])["Close"]

    #Dropping any Nan values
    df.dropna(inplace=True)
    
    sequence=[]
    # We want to scale the data except the label part since it is already 0 and 1
    temp=df.loc[:, df.columns != 'Label']
#     temp=scale_data(temp)
    temp = scaler.fit_transform(temp)
    # print(f"temp{temp[:30]}")
    for i in range (len(temp)-SERIES_LENGTH):
       sequence.append([np.array(temp[i:i+SERIES_LENGTH]),df.iloc[i+SERIES_LENGTH,-1]]) # iloc part is to take last column data i.e. labels

    np.random.shuffle(sequence)

    #Now we will count the sells and buys to balance the data
    # Algorithm : whichever count is less, we will take up the data upto that
    X=[]
    y=[]
    buy=[]
    sell=[]
    for seq ,label in sequence:
        if label == 0:
            sell.append([seq,label])
        else:
            buy.append([seq,label])
            
    # print(f"buy :{buy[:10]}")
    # print(f"sell :{sell[:10]}")
    
    buys=len(buy)
    sells=len(sell)
    print(f"original buys:{buys} original sells:{sells}")
    if(buys<sells):
        buy=buy[:buys]
        sell=sell[:buys]
    else:
        buy=buy[:sells]
        sell=sell[:sells]

    print(f"buys:{len(buy)} sells:{len(sell)}")
    # Concat the buys an sells and shuffle it out again
    sequence=buy+sell

    np.random.shuffle(sequence)


    for seq ,label in sequence:
        X.append(seq)
        y.append(label)

    return np.array(X),np.array(y)

In [ ]:
df.shape

In [ ]:
process_data(df)

In [ ]:
df['Label'].value_counts()

In [ ]:
# Plot target variable
plt.figure(figsize=(8,4))
sns.countplot('Label', data=df)
plt.title('Target Variable Count')
plt.show()

In [ ]:
training_size=0.8

In [ ]:
spilt_point=int(training_size*len(df))

In [ ]:
#splitting data for training and testing in ratio 80:20
train_df=df[:spilt_point]
test_df=df[spilt_point:]

In [ ]:
import warnings
warnings.filterwarnings("ignore")

train_x,train_y=process_data(train_df)

test_x,test_y=process_data(test_df)

In [ ]:
print('X_train :',train_x.shape)
print('y_train :',train_y.shape)
print('X_test :',test_x.shape)
print('y_test :',test_y.shape)

In [ ]:
def build_model_CNN():

    model=Sequential()
    
    model.add(Conv1D(256, kernel_size=2, activation='tanh', input_shape=(train_x.shape[1:])))
    model.add(Dropout(0.2))
    
    model.add(Conv1D(128, kernel_size=2, activation='tanh', input_shape=(train_x.shape[1:])))
    model.add(Dropout(0.2))
    
    model.add(Conv1D(64, kernel_size=2, activation='tanh', input_shape=(train_x.shape[1:])))
    model.add(Dropout(0.2))
    
    model.add(MaxPooling1D(2))
    model.add(Flatten())

    model.add(Dense(32, kernel_initializer="uniform", activation='tanh'))
    model.add(Dense(1, kernel_initializer="uniform", activation='sigmoid'))


    model.compile(loss='binary_crossentropy',optimizer="adam",metrics=['accuracy'])
    history=model.fit(train_x, train_y, batch_size=96, epochs=100, validation_data=(test_x,test_y))
    score=model.evaluate(test_x,test_y)
    cnn_pred=model.predict(test_x) > 0.5

    print("Validation accuracy percentage",score[1])
    print("Validation loss percentage",score[0])
    print("Precision score", precision_score(test_y, cnn_pred, average='macro'))
    print("Accuracy score", accuracy_score(test_y, cnn_pred, normalize=True))
    
    return model, cnn_pred

In [ ]:
model_CNN, cnn_pred = build_model_CNN()

Epoch 59/100
40/40 [==============================] - 0s 7ms/step - loss: 0.2267 - accuracy: 0.9022 - val_loss: 0.3262 - val_accuracy: 0.8668
Epoch 60/100
40/40 [==============================] - 0s 7ms/step - loss: 0.2233 - accuracy: 0.9072 - val_loss: 0.4132 - val_accuracy: 0.8473
Epoch 61/100
40/40 [==============================] - 0s 7ms/step - loss: 0.2263 - accuracy: 0.9001 - val_loss: 0.4708 - val_accuracy: 0.8053
Epoch 62/100
40/40 [==============================] - 0s 8ms/step - loss: 0.2308 - accuracy: 0.8963 - val_loss: 0.4403 - val_accuracy: 0.8186
Epoch 63/100
40/40 [==============================] - 0s 7ms/step - loss: 0.2521 - accuracy: 0.8859 - val_loss: 0.3965 - val_accuracy: 0.8350
Epoch 64/100
40/40 [==============================] - 0s 7ms/step - loss: 0.2376 - accuracy: 0.8963 - val_loss: 0.4872 - val_accuracy: 0.8023
Epoch 65/100
40/40 [==============================] - 0s 8ms/step - loss: 0.2310 - accuracy: 0.8955 - val_loss: 0.2778 - val_accuracy: 0.8852
Epoch 

In [ ]:
mat = confusion_matrix(test_y, cnn_pred)
labels = ['Legitimate', 'Fraudulent']
 
sns.heatmap(mat, square=True, annot=True, fmt='d', cbar=False, cmap='Blues',
            xticklabels=labels, yticklabels=labels)
 
plt.xlabel('Predicted label')
plt.ylabel('Actual label')

In [ ]:
print(classification_report(test_y, cnn_pred))